# Desafio de Regressão - Previsão de Custos de Seguro Médico

Este notebook implementa uma solução para prever custos de seguro médico com base em características do beneficiário.

**Objetivos:**
1. Carregar e explorar o dataset.
2. Pré-processar os dados (tratamento de variáveis categóricas).
3. Treinar dois modelos de regressão distintos.
4. Utilizar `VotingRegressor` para combinar as predições.
5. Avaliar o desempenho.
6. Salvar o modelo para uso em API.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Configuração de visualização
sns.set(style="whitegrid")

## 1. Carregamento dos Dados
Utilizaremos o dataset 'Medical Cost Personal Datasets'.

In [ ]:
# URL direta do dataset
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

# Visualizar as primeiras linhas
df.head()

In [ ]:
# Informações sobre o dataset
df.info()

## 2. Pré-processamento
Separar features (X) e target (y), e preparar o pipeline de transformação para variáveis categóricas.

In [ ]:
# Separar Features e Target
X = df.drop('charges', axis=1)
y = df['charges']

# Identificar colunas categóricas e numéricas
categorical_features = ['sex', 'smoker', 'region']
numerical_features = ['age', 'bmi', 'children']

# Criar transformadores
# OneHotEncoder para categóricas, StandardScaler para numéricas (opcional, mas bom para alguns modelos)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

## 3. Treinamento dos Modelos
Vamos treinar dois modelos base: Random Forest e Gradient Boosting.

In [ ]:
# Modelo 1: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Modelo 2: Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

# Pipeline individual (apenas para avaliação isolada se necessário)
pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor), ('model', rf)])
pipeline_gb = Pipeline(steps=[('preprocessor', preprocessor), ('model', gb)])

# Treinar e avaliar individualmente (opcional, para comparação)
pipeline_rf.fit(X_train, y_train)
print("Random Forest R2:", pipeline_rf.score(X_test, y_test))

pipeline_gb.fit(X_train, y_train)
print("Gradient Boosting R2:", pipeline_gb.score(X_test, y_test))

## 4. Voting Regressor
Combinando os modelos para tentar obter uma predição mais robusta.

In [ ]:
# Criar o Voting Regressor
voting_model = VotingRegressor(estimators=[
    ('rf', rf),
    ('gb', gb)
])

# Criar o Pipeline final com o Voting Regressor
final_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', voting_model)])

# Treinar o modelo final
final_pipeline.fit(X_train, y_train)

## 5. Avaliação do Modelo Final

In [ ]:
# Predições
y_pred = final_pipeline.predict(X_test)

# Métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

# Plot: Real vs Predito
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Valor Real')
plt.ylabel('Valor Predito')
plt.title('Real vs Predito - Voting Regressor')
plt.show()

## 6. Salvar o Modelo
Salvaremos o pipeline completo (pré-processamento + modelo) para facilitar o uso na API.

In [ ]:
import joblib

# Salvar o modelo treinado
model_filename = 'voting_model.joblib'
joblib.dump(final_pipeline, model_filename)
print(f"Modelo salvo como {model_filename}")

# Exemplo de como carregar e prever (teste rápido)
loaded_model = joblib.load(model_filename)

# Exemplo de input (um dicionário convertido para DataFrame)
sample_data = pd.DataFrame([{
    'age': 30,
    'sex': 'male',
    'bmi': 25.5,
    'children': 0,
    'smoker': 'no',
    'region': 'southwest'
}])

prediction = loaded_model.predict(sample_data)
print(f"Previsão de custo para o exemplo: {prediction[0]:.2f}")